In [1]:
# !pip install plotly numpy pandas matplotlib
# !pip install --upgrade nbformat

In [2]:
OriginalProjectPath = "../../2018 FreudMeOutProject"
SecondProjectPath = "../../2019 Freud2.0"
MobileProjectPath = "../../2020 Mobile"
VrProjectPath = "../../2023 affective-game-vr-main"
TestProjectPath = "../."

In [3]:
def getProjectNameFromPath(path):
    return path.split("/")[-1]

In [4]:
import os
import plotly.graph_objects as go
from collections import defaultdict

In [5]:
def human_readable_size(size_bytes):
    """
    Convert a file size in bytes to a human-readable string format.
    e.g., 2048 -> '2.0 KB'
    """
    if size_bytes == 0:
        return "0 B"
    units = ["B", "KB", "MB", "GB", "TB"]
    i = 0
    while size_bytes >= 1024 and i < len(units) - 1:
        size_bytes /= 1024.0
        i += 1
    return f"{size_bytes:.1f} {units[i]}"

In [6]:
class DirectoryNode:
    # Class to handle node in directory structure
    # Class should have absolute path, relative path, reference to parent node, reference to children nodes, list of files,
    # and cached size. If size is not cached, it should be calculated on demand. Size equals to sum of all files in the node and all children nodes.

    def __init__(self, absolute_path, relative_path="", parent=None):
        self.absolute_path = absolute_path
        self.relative_path = relative_path
        self.parent = parent
        self.children = []
        self.files = []
        self._cached_size = None
        self._cached_file_count = None

    def get_name(self):
        return os.path.basename(self.absolute_path)
    
    def add_file(self, file_name):
        self.files.append(file_name)

    def add_child(self, child_node):
        self.children.append(child_node)

    def get_size(self):
        if self._cached_size is not None:
            return self._cached_size
        total_size = sum(os.path.getsize(os.path.join(self.absolute_path, f)) for f in self.files)
        for child in self.children:
            total_size += child.get_size()
        self._cached_size = total_size
        return total_size
    
    #element count is number of files in the node and all children nodes
    def get_file_count(self):
        if self._cached_file_count is not None:
            return self._cached_file_count
        total_count = len(self.files)
        for child in self.children:
            total_count += child.get_file_count()
        self._cached_file_count = total_count
        return total_count
    
    #preety print all properties, by printing all properties of the class, and tabbed all properties of the children
    def pretty_print(self, indent=0):
        print(" " * indent + f"Node: {self.get_name()} (Size: {human_readable_size(self.get_size())})")
        for file in self.files:
            print(" " * (indent + 2) + f"File: {file}")
        for child in self.children:
            child.pretty_print(indent + 2)


In [7]:
def generate_directory_tree(base_path):
    """
    Generates a directory tree structure starting from the base path.
    Returns the root DirectoryNode.
    """
    absolute_path = os.path.abspath(base_path)
    # print(f"Generating directory tree for: {absolute_path}")
    root_node = DirectoryNode(absolute_path=absolute_path, relative_path=os.path.basename(absolute_path), parent=None)
    nodes = {base_path: root_node}

    for root, dirs, files in os.walk(base_path):
        current_node = nodes[root]
        for file in files:
            current_node.add_file(file)
        for dir_name in dirs:
            dir_path = os.path.join(root, dir_name)
            relative_path = os.path.relpath(dir_path, base_path)
            child_node = DirectoryNode(absolute_path=dir_path, relative_path=relative_path, parent=current_node)
            current_node.add_child(child_node)
            nodes[dir_path] = child_node

    return root_node

In [8]:
#create and enum to switch between size and file count
class DisplayMode:
    SIZE = "size"
    FILE_COUNT = "file_count"

In [9]:
def map(inmin, inmax, outmin, outmax, value):
    return (value - inmin) / (inmax - inmin) * (outmax - outmin) + outmin

In [10]:
# def generate_sankey_from_tree_dfs(root, display_mode=DisplayMode.FILE_COUNT, max_depth=None):
#     """
#     Converts a DirectoryNode tree into Plotly Sankey data.
#     Uses `relative_path` for unique indexing and `get_name()` for display.
#     Optionally limits the depth of the tree with max_depth.

#     Returns:
#         labels (List[str]): Display names (from get_name())
#         source (List[int]): Parent node indices
#         target (List[int]): Child node indices
#         value (List[int]): Flow values (in bytes or file count)
#     """

#     if max_depth is not None and max_depth < 2:
#         raise ValueError("max_depth must be at least 2 to include root and children nodes.")

#     labels = []
#     index_map = {}  # Key: relative_path, Value: index in labels
#     source = []
#     target = []
#     value = []
#     depth_to_nodeCount = defaultdict(int)  # For manual y-coordinates if needed

#     def add_label(node):
#         rel = node.relative_path or "root"
#         if rel not in index_map:
#             index_map[rel] = len(labels)
#             appendix_core = human_readable_size(node.get_size()) if display_mode == DisplayMode.SIZE else str(node.get_file_count())
#             labels.append(node.get_name() + " (" + appendix_core + ")")
#         return index_map[rel]

#     def add_node(node, depth=0):
#         depth_to_nodeCount[depth] += 1
#         if max_depth is not None and depth > max_depth-2:
#             return
#         parent_idx = add_label(node)
#         for child in node.children:
#             child_idx = add_label(child)
#             source.append(parent_idx)
#             target.append(child_idx)
#             value.append(child.get_size() if display_mode == DisplayMode.SIZE else child.get_file_count())
#             add_node(child, depth + 1) # this here means the algoritm is DFS

#     add_node(root)

#     return labels, source, target, value, depth_to_nodeCount


In [114]:
from math import sqrt

def process_x_offsets(inX):
    return inX**1.3

def generate_offsets_from_depths(depth_to_nodeCount):
    keys = sorted(depth_to_nodeCount.keys())
    x_offsets = []
    y_offsets =[]
    for depth in keys:
        count = depth_to_nodeCount[depth]
        x_offsets.extend([depth / (len(keys) - 1)] * count)
        y_offsets.extend([i / (max(1,count-1)) for i in range(0, count)])
    # x_offsets = list(sqrt(x) for x in x_offsets)
    #replace 0 with 0.01 to avoid division by zero 
    x_offsets = [0.01 if x == 0 else x for x in x_offsets]
    y_offsets = [0.01 if y == 0 else y for y in y_offsets]
    x_offsets = [0.99 if x > 0.99 else x for x in x_offsets]  # Cap at 0.99
    y_offsets = [0.99 if y > 0.99 else y for y in y_offsets]  # Cap at 0.99
    x_offsets = list(process_x_offsets(x) for x in x_offsets)
    print(f"X offsets: {x_offsets}")
    print(f"Y offsets: {y_offsets}")
    return x_offsets, y_offsets

In [115]:
def generate_sankey_from_tree_bfs(root, display_mode=DisplayMode.FILE_COUNT, max_depth=None):
    if max_depth is not None and max_depth < 2:
        raise ValueError("max_depth must be at least 2 to include root and children nodes.")

    labels = []
    index_map = {}
    source = []
    target = []
    value = []
    depth_to_nodeCount = defaultdict(int)  # For manual y-coordinates if needed

    def add_label(node):
        rel = node.relative_path or "root"
        if rel not in index_map:
            index_map[rel] = len(labels)
            appendix = human_readable_size(node.get_size()) if display_mode == DisplayMode.SIZE else str(node.get_file_count())
            labels.append(node.get_name() + " (" + appendix + ")")
        return index_map[rel]

    queue = [(root, 0)]
    while queue:
        node, depth = queue.pop(0)
        depth_to_nodeCount[depth] += 1
        if max_depth is not None and depth > max_depth - 2:
            continue

        parent_idx = add_label(node)
        for child in node.children:
            child_idx = add_label(child)
            source.append(parent_idx)
            target.append(child_idx)
            value.append(child.get_size() if display_mode == DisplayMode.SIZE else child.get_file_count())
            queue.append((child, depth + 1))

    return labels, source, target, value, depth_to_nodeCount


In [ ]:
import numpy as np
def plot_sankey(labels, source, target, value, depth_to_nodeCount , title="Directory Size Sankey Diagram"):
    print(depth_to_nodeCount)
    x_offsets, y_offsets = generate_offsets_from_depths(depth_to_nodeCount)
    # print labels
    print(f"Labels: {labels}")
    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(
            label=labels,
            align="left",
            x=x_offsets,
            y=y_offsets,
            ),
        link=dict(
            source=source,
            target=target,
            value=value,
        )
    ))
    fig.update_layout(title_text=title, font_size=14,height=600, width=1500)
    fig.show()

In [117]:
plot_sankey(*generate_sankey_from_tree_bfs(generate_directory_tree(TestProjectPath), display_mode=DisplayMode.SIZE,max_depth=4),title=getProjectNameFromPath(TestProjectPath) + " ( Folder sizes )")
plot_sankey(*generate_sankey_from_tree_bfs(generate_directory_tree(TestProjectPath), display_mode=DisplayMode.FILE_COUNT,max_depth=4),title=getProjectNameFromPath(TestProjectPath) + " ( File count )")

defaultdict(<class 'int'>, {0: 1, 1: 2, 2: 3, 3: 6})
X offsets: [0.00251188643150958, 0.2397410311082881, 0.2397410311082881, 0.5903116621970373, 0.5903116621970373, 0.5903116621970373, 0.9870195456944257, 0.9870195456944257, 0.9870195456944257, 0.9870195456944257, 0.9870195456944257, 0.9870195456944257]
Y offsets: [0.01, 0.01, 0.99, 0.01, 0.5, 0.99, 0.01, 0.2, 0.4, 0.6, 0.8, 0.99]
Labels: ['Extras (35.5 MB)', 'Assets (35.4 MB)', 'Initial project sankey analyzer (74.0 KB)', 'Models (5.7 MB)', 'PostProcessing (29.4 MB)', 'Standard Assets (253.4 KB)', 'Characters (2.1 MB)', 'Environment (3.6 MB)', 'Textures (29.4 MB)', 'CrossPlatformInput (183.2 KB)', 'PhysicsMaterials (2.6 KB)', 'Utility (67.7 KB)']


defaultdict(<class 'int'>, {0: 1, 1: 2, 2: 3, 3: 6})
X offsets: [0.00251188643150958, 0.2397410311082881, 0.2397410311082881, 0.5903116621970373, 0.5903116621970373, 0.5903116621970373, 0.9870195456944257, 0.9870195456944257, 0.9870195456944257, 0.9870195456944257, 0.9870195456944257, 0.9870195456944257]
Y offsets: [0.01, 0.01, 0.99, 0.01, 0.5, 0.99, 0.01, 0.2, 0.4, 0.6, 0.8, 0.99]
Labels: ['Extras (82)', 'Assets (81)', 'Initial project sankey analyzer (1)', 'Models (19)', 'PostProcessing (10)', 'Standard Assets (51)', 'Characters (4)', 'Environment (15)', 'Textures (10)', 'CrossPlatformInput (22)', 'PhysicsMaterials (6)', 'Utility (23)']


In [96]:
import plotly.graph_objects as go

labels = [
    # Column 1
    "A1", "A2", "A3",      # 0, 1, 2
    # Column 2
    "B1", "B2", "B3",      # 3, 4, 5
    # Column 3
    "C1", "C2", "C3",      # 6, 7, 8
    # Column 4 (D3 on top, D1 on bottom)
    "D1", "D2", "D3",      # 9, 10, 11
    # Column 5
    "E1", "E2", "E3"       # 12, 13, 14
]

# Define source-target flow (1-to-1, unmerged)
sources = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
targets = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
values  = [1] * len(sources)

# Manually define x and y coordinates for each node (0 to 1 scale)
x = [
    0.01, 0.01, 0.01,   # A1-A3
    0.25, 0.25, 0.25,  # B1-B3
    0.5, 0.5, 0.5,     # C1-C3
    0.75, 0.75, 0.75,  # D1-D3
    1.0, 1.0, 1.0      # E1-E3
]

# Default top-down order for A, B, C, E
y = [
    1.0, 0.5, 0.2,   # A1-A3
    1.0, 0.5, 0.2,   # B1-B3
    1.0, 0.5, 0.2,   # C1-C3
    1.0, 0.5, 0.2,   # D1-D3 (reversed y to show D3 at top, D1 at bottom)
    1.0, 0.5, 0.2    # E1-E3
]
ys= np.random.rand(len(labels))
from math import floor
ys=list(floor(x*100)/100 for x in ys)
print(f"Y offsets: {ys}")
fig = go.Figure(data=[go.Sankey(
    arrangement="snap",  # Needed when using manual x/y
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=labels,
        x=x,
        # set y to random values, use numpy to generate random values
        y=ys
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values
    )
)])

fig.update_layout(title_text="Sankey Diagram with Reversed D Column (D3 to D1)", font_size=10)
fig.show()


Y offsets: [0.09, 0.3, 0.41, 0.03, 0.61, 0.01, 0.58, 0.8, 0.84, 0.85, 0.19, 0.04, 0.33, 0.85, 0.53]
